<a href="https://colab.research.google.com/github/JasonPeer/inventory_volatility_analysis/blob/main/notebooks/01_data_transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# Objective:
# Convert the raw CA_1 wide-format sales data into a long-format daily demand dataset
# for subsequent volatility measurement and inventory simulation.


#Import Environment
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('/content/drive/MyDrive/YBF_PP/sales_train_validation.csv')

df.shape


#Check the validation of our data
df_ca1 = df[df["store_id"] == "CA_1"].copy()

df_ca1.shape

df_ca1["store_id"].unique()


day_cols = [col for col in df_ca1.columns if col.startswith("d_")]

len(day_cols), day_cols[:5], day_cols[-5:]


#Melt().Transform wide format into long format
df_long = df_ca1.melt(
    id_vars=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
    value_vars=day_cols,
    var_name="day",
    value_name="demand"
)

df_long.shape

df_long.head()


#Import Calendar Data
calendar = pd.read_csv('/content/drive/MyDrive/YBF_PP/calendar.csv')
calendar.head()

#Cleaning Calendar data
calendar_small = calendar[["d", "date"]].copy()
calendar_small = calendar_small.rename(columns={"d": "day"})
calendar_small.head()

#Merging the two csv
df_long = df_long.merge(calendar_small, on="day", how="left")
df_long.head()

#Sorting the columns
df_long = df_long[
    ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id", "date", "day", "demand"]
]
df_long.head()

#Saving the processed long-format dataset
df_long.to_csv('/content/drive/MyDrive/YBF_PP/ca1_long_sales.csv', index=False)

#The final doc is too big to review, so I decide to delete some of the columns
df_long_small = df_long[["item_id", "dept_id", "cat_id", "store_id", "date", "demand"]].copy()

df_long_small.to_csv('/content/drive/MyDrive/YBF_PP/ca1_long_sales_small.csv', index=False)

#Also by doing research I find out that .parquet format is lighter than .csv.
df_long_small.to_parquet('/content/drive/MyDrive/YBF_PP/ca1_long_sales_small.parquet', index=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
